# Integrate synthetic Litware capacity

Upload the four Litware CSV files to Files/litware in the attached Caldova Lakehouse. This notebook validates all inputs before writing five managed Delta tables. It adds a stable qualification key and evaluates complete planning windows. It doesn't update Caldova source facts.

In [ ]:
import csv
from collections import defaultdict
from datetime import date
from pathlib import Path

def load_litware(data_root, valid_product_ids, deadline):
    tables = {}
    counts = {'litware_plant': 3, 'litware_product_mapping': 4,
              'litware_capacity_plan': 18, 'litware_product_qualification': 6}
    for name, count in counts.items():
        with (Path(data_root) / f'{name}.csv').open(encoding='utf-8-sig', newline='') as stream:
            rows = list(csv.DictReader(stream))
        if len(rows) != count or any(None in row or None in row.values() for row in rows):
            raise ValueError(f'{name}: expected {count} complete CSV records.')
        tables[name] = rows

    def index(rows, columns):
        result = {tuple(row[column] for column in columns): row for row in rows}
        if len(result) != len(rows) or any(not all(key) for key in result):
            raise ValueError(f'Duplicate or empty key: {columns}')
        return result

    plants = index(tables['litware_plant'], ['LitwarePlantID'])
    products = index(tables['litware_product_mapping'], ['LitwareProductID'])
    qualifications = index(tables['litware_product_qualification'], ['LitwarePlantID', 'LitwareProductID'])
    index(tables['litware_capacity_plan'], ['LitwareCapacityPlanID'])
    for product in products.values():
        mapped = product['CaldovaProductID']
        if product['MappingStatus'] not in {'Approved', 'Unmapped'}:
            raise ValueError('Unsupported mapping status.')
        if (product['MappingStatus'] == 'Approved' and mapped not in valid_product_ids) or (product['MappingStatus'] == 'Unmapped' and mapped):
            raise ValueError('Invalid Caldova product mapping.')
        product['CaldovaProductID'] = mapped or None
    for pair, qualification in qualifications.items():
        if (pair[0],) not in plants or (pair[1],) not in products:
            raise ValueError('Orphan qualification.')
        qualification['QualificationID'] = '|'.join(pair)
        for column in ['ValidFromDate', 'ValidToDate', 'EarliestReadyDate']:
            qualification[column] = date.fromisoformat(qualification[column])
        qualification['OpenCriticalFindings'] = int(qualification['OpenCriticalFindings'])
        if qualification['ValidFromDate'] > qualification['ValidToDate'] or qualification['OpenCriticalFindings'] < 0:
            raise ValueError('Invalid qualification dates or findings.')
        if qualification['QualificationStatus'] not in {'Approved', 'Pending', 'Rejected'}:
            raise ValueError('Unsupported qualification status.')

    windows = defaultdict(list)
    assessments = []
    for row in tables['litware_capacity_plan']:
        pair = row['LitwarePlantID'], row['LitwareProductID']
        if pair not in qualifications:
            raise ValueError('Capacity has no matching qualification.')
        plant, product, qualification = plants[(pair[0],)], products[(pair[1],)], qualifications[pair]
        for column in ['WindowStartDate', 'WindowEndDate']:
            row[column] = date.fromisoformat(row[column])
        for column in ['MaxCapacityUnits', 'CommittedCapacityUnits', 'AvailableCapacityUnits']:
            row[column] = int(row[column])
        start, end = row['WindowStartDate'], row['WindowEndDate']
        if start > end or any(start <= prior_end and end >= prior_start for prior_start, prior_end in windows[pair]):
            raise ValueError('Invalid or overlapping capacity windows.')
        windows[pair].append((start, end))
        if not 0 <= row['CommittedCapacityUnits'] <= row['MaxCapacityUnits'] or row['AvailableCapacityUnits'] != row['MaxCapacityUnits'] - row['CommittedCapacityUnits']:
            raise ValueError('Invalid capacity arithmetic.')
        if row['CapacityUnit'] != 'FinishedProductUnits':
            raise ValueError('Unsupported capacity unit.')
        reasons = []
        if product['MappingStatus'] != 'Approved': reasons.append('Unmapped product')
        if plant['GMPStatus'] != 'Compliant': reasons.append('Plant not compliant')
        if qualification['QualificationStatus'] != 'Approved': reasons.append('Qualification not approved')
        if qualification['OpenCriticalFindings'] != 0: reasons.append('Open critical findings')
        if start < qualification['ValidFromDate'] or end > qualification['ValidToDate']: reasons.append('Qualification outside window')
        if start < qualification['EarliestReadyDate']: reasons.append('Not ready for window')
        if start < date(2026, 11, 1) or end > deadline: reasons.append('Outside launch planning window')
        assessments.append({**row, 'QualificationID': qualification['QualificationID'],
                            'CaldovaProductID': product['CaldovaProductID'], 'PlantType': plant['PlantType'],
                            'EligibleCapacityUnits': 0 if reasons else row['AvailableCapacityUnits'],
                            'ExclusionReason': '; '.join(reasons) or 'None'})
    tables['litware_capacity_assessment'] = assessments
    totals = defaultdict(int)
    for row in assessments:
        totals[(row['LitwarePlantID'], row['LitwareProductID'])] += row['AvailableCapacityUnits']
    expected = {('LPL001', 'LPR001'): 12000, ('LPL002', 'LPR001'): 24000,
                ('LPL003', 'LPR001'): 18000, ('LPL001', 'LPR002'): 9000,
                ('LPL002', 'LPR003'): 6000, ('LPL003', 'LPR004'): 3000}
    eligible = [row for row in assessments if row['EligibleCapacityUnits'] > 0]
    if dict(totals) != expected or sum(row['EligibleCapacityUnits'] for row in eligible) != 12000 or any((row['LitwarePlantID'], row['LitwareProductID'], row['PlantType']) != ('LPL001', 'LPR001', 'Packaging') for row in eligible):
        raise ValueError('Synthetic fixture differs from the expected Litware scenario.')
    return tables

In [ ]:
from pyspark.sql import functions as F

product_ids = {row.ProductID for row in spark.table('dbo.product').select('ProductID').collect()}
launches = spark.table('dbo.launch_program').filter(F.col('LaunchProgramID') == 'LP001').collect()
if len(launches) != 1 or str(launches[0].TargetLaunchDate)[:10] != '2026-11-26':
    raise ValueError('Expected the supplied LP001 launch deadline of November 26, 2026.')
tables = load_litware('/lakehouse/default/Files/litware', product_ids, date(2026, 11, 26))
print('Validated 31 synthetic source records before writing any Litware tables.')

In [ ]:
capacity_schema = 'LitwareCapacityPlanID string, LitwarePlantID string, LitwareProductID string, WindowStartDate date, WindowEndDate date, MaxCapacityUnits long, CommittedCapacityUnits long, AvailableCapacityUnits long, CapacityUnit string'
schemas = {
    'litware_plant': 'LitwarePlantID string, PlantName string, Country string, PlantType string, GMPStatus string',
    'litware_product_mapping': 'LitwareProductID string, LitwareProductName string, CaldovaProductID string, MappingStatus string',
    'litware_capacity_plan': capacity_schema,
    'litware_product_qualification': 'LitwarePlantID string, LitwareProductID string, QualificationStatus string, ValidFromDate date, ValidToDate date, OpenCriticalFindings int, EarliestReadyDate date, QualificationID string',
    'litware_capacity_assessment': capacity_schema + ', QualificationID string, CaldovaProductID string, PlantType string, EligibleCapacityUnits long, ExclusionReason string',
}
for name, records in tables.items():
    frame = spark.createDataFrame(records, schema=schemas[name])
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'dbo.{name}')
    print(f'Wrote {name}: {frame.count()} rows')
display(spark.table('dbo.litware_capacity_assessment').groupBy('CaldovaProductID', 'LitwarePlantID', 'PlantType', 'ExclusionReason').agg(F.sum('AvailableCapacityUnits').alias('AvailableUnits'), F.sum('EligibleCapacityUnits').alias('EligibleUnits')))
print('Validation passed: 31 source records; 18 assessed windows; 12,000 eligible planned Packaging units for P101.')